# Integration de la dynamique hamiltonienne

On etudie $\dot q=p$, $\dot p=-V'(q)$ et $H(q,p)=p^2/2+V(q)$ avec $M=1$.

Potentiels : $V(q)=q^2/2$ et $V(q)=(q^2-1)^2$. L'objectif est d'observer l'Hamiltonien en temps long pour plusieurs pas $\Delta t$.

## Conservation de l'Hamiltonien

Par la regle de la chaine,

$$\frac{dH}{dt}=V'(q)\dot q+p\dot p=V'(q)p-pV'(q)=0.$$

L'Hamiltonien est donc constant pour la dynamique exacte.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

def V_quadratic(q):
    return 0.5 * q**2

def grad_quadratic(q):
    return q

def V_double_well(q):
    return (q**2 - 1.0)**2

def grad_double_well(q):
    return 4.0 * q * (q**2 - 1.0)

def H(q, p, potential):
    return 0.5 * p**2 + potential(q)


## Schemas numeriques

Euler symplectique A : $p_{n+1}=p_n-\Delta t V'(q_n)$ puis $q_{n+1}=q_n+\Delta t p_{n+1}$.

Verlet : $p_{n+1/2}=p_n-\Delta t V'(q_n)/2$, $q_{n+1}=q_n+\Delta t p_{n+1/2}$, puis $p_{n+1}=p_{n+1/2}-\Delta t V'(q_{n+1})/2$.

In [ ]:
def symplectic_euler(q0, p0, dt, T, grad_V):
    n = int(round(T / dt))
    t = dt * np.arange(n + 1)
    q, p = np.empty(n + 1), np.empty(n + 1)
    q[0], p[0] = q0, p0
    for k in range(n):
        p[k + 1] = p[k] - dt * grad_V(q[k])
        q[k + 1] = q[k] + dt * p[k + 1]
    return t, q, p

def verlet(q0, p0, dt, T, grad_V):
    n = int(round(T / dt))
    t = dt * np.arange(n + 1)
    q, p = np.empty(n + 1), np.empty(n + 1)
    q[0], p[0] = q0, p0
    for k in range(n):
        p_half = p[k] - 0.5 * dt * grad_V(q[k])
        q[k + 1] = q[k] + dt * p_half
        p[k + 1] = p_half - 0.5 * dt * grad_V(q[k + 1])
    return t, q, p


## Potentiel quadratique : evolution en temps long

Pour $V(q)=q^2/2$, on a $\ddot q+q=0$, donc $q(t)=q_0\cos(t)+p_0\sin(t)$ et $p(t)=-q_0\sin(t)+p_0\cos(t)$. Ainsi $H(t)=H(0)$ exactement.

Pour Euler symplectique et Verlet, l'energie numerique reste bornee et oscille si $0<\Delta t<2$. Euler symplectique a une erreur d'energie en $O(\Delta t)$, tandis que Verlet est d'ordre 2 et a une erreur en $O(\Delta t^2)$. Pour $\Delta t\geq 2$, le cas quadratique devient instable.

In [ ]:
q0, p0, T = 1.0, 0.0, 2000.0
dts = [0.1, 0.2, 0.4, 1.0]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for dt in dts:
    t, qe, pe = symplectic_euler(q0, p0, dt, T, grad_quadratic)
    t, qv, pv = verlet(q0, p0, dt, T, grad_quadratic)
    ax[0].plot(t, H(qe, pe, V_quadratic) - H(qe[0], pe[0], V_quadratic), label=f'dt={dt}')
    ax[1].plot(t, H(qv, pv, V_quadratic) - H(qv[0], pv[0], V_quadratic), label=f'dt={dt}')
ax[0].set_title('Euler symplectique : H(t)-H(0)')
ax[1].set_title('Verlet : H(t)-H(0)')
for a in ax:
    a.set_xlabel('t')
    a.set_ylabel('variation d energie')
    a.legend()
plt.tight_layout()


## Potentiel double puits

On repete l'etude avec $V(q)=(q^2-1)^2$, qui possede deux minima en $q=\pm1$.

In [ ]:
q0, p0, T = 0.0, 1.0, 500.0
dts = [0.02, 0.05, 0.1]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for dt in dts:
    t, qe, pe = symplectic_euler(q0, p0, dt, T, grad_double_well)
    t, qv, pv = verlet(q0, p0, dt, T, grad_double_well)
    He = H(qe, pe, V_double_well)
    Hv = H(qv, pv, V_double_well)
    ax[0].plot(t, He - He[0], label=f'Euler dt={dt}')
    ax[0].plot(t, Hv - Hv[0], '--', label=f'Verlet dt={dt}')
    ax[1].plot(qv, pv, label=f'dt={dt}')
ax[0].set_title('Double puits : variation de H')
ax[0].set_xlabel('t')
ax[0].set_ylabel('H(t)-H(0)')
ax[0].legend(ncol=2)
ax[1].set_title('Double puits : portrait de phase')
ax[1].set_xlabel('q')
ax[1].set_ylabel('p')
ax[1].legend()
plt.tight_layout()


## Conclusion

La dynamique exacte conserve l'Hamiltonien. Les schemas symplectiques limitent la derive de l'energie sur les temps longs ; Verlet est plus precis qu'Euler symplectique car il est d'ordre 2.